<a href="https://colab.research.google.com/github/Shruti903/BIA-PROJECT-2026/blob/main/BIA_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✈️ Airline Revenue Intelligence System
### Business Intelligence & Predictive Analytics on 300K+ Flight Tickets

**Dataset:** [Kaggle – airlines-flights-data](https://www.kaggle.com/datasets/rohitgrewal/airlines-flights-data) (India domestic flights, prices in ₹)

**Pipeline:** Load → Clean → Feature Engineering → KPIs → BI Dashboards (Plotly) → Revenue Insights → ML Price Prediction → Route Clustering → Anomaly Detection → Export for Power BI → Streamlit Dashboard

> Run the cells from top to bottom (Runtime → Run all).

## STEP 0 – Install & Import

In [ ]:
!pip -q install kagglehub plotly scikit-learn streamlit
import pandas as pd, numpy as np, glob, os, warnings
import plotly.express as px
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Libraries ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 49.9 MB/s eta 0:00:00
Libraries ready


## STEP 1 – Load the dataset
**Option A (automatic):** downloads from Kaggle with `kagglehub` (no API key needed for public datasets).
**Option B (manual):** download the CSV from Kaggle, then run the upload cell below instead.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("rohitgrewal/airlines-flights-data")
csv_file = glob.glob(os.path.join(path, "*.csv"))[0]
df = pd.read_csv(csv_file)
print(csv_file)
df.head()

100%|██████████| 2.33M/2.33M [00:00<00:00, 43.2MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/rohitgrewal/airlines-flights-data/versions/1/airlines_flights_data.csv


,index,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


In [ ]:
# OPTION B – only run this if Option A fails
# from google.colab import files
# up = files.upload()
# df = pd.read_csv(list(up.keys())[0])

## STEP 2 – Understand & clean the data

In [ ]:
print(df.shape)
df.info()
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
df.describe(include='all').T

(300153, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   index             300153 non-null  int64  
 1   airline           300153 non-null  object 
 2   flight            300153 non-null  object 
 3   source_city       300153 non-null  object 
 4   departure_time    300153 non-null  object 
 5   stops             300153 non-null  object 
 6   arrival_time      300153 non-null  object 
 7   destination_city  300153 non-null  object 
 8   class             300153 non-null  object 
 9   duration          300153 non-null  float64
 10  days_left         300153 non-null  int64  
 11  price             300153 non-null  int64  
dtypes: float64(1), int64(3), object(8)
memory usage: 27.5+ MB

Missing values:
 index               0
airline             0
flight              0
source_city         0
departure_time      0
stops        

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
index,"300,153.00",NaN,NaN,NaN,"150,076.00","86,646.85",0.00,"75,038.00","150,076.00","225,114.00","300,152.00"
airline,300153,6,Vistara,127859,NaN,NaN,NaN,NaN,NaN,NaN,NaN
flight,300153,1561,UK-706,3235,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_city,300153,6,Delhi,61343,NaN,NaN,NaN,NaN,NaN,NaN,NaN
departure_time,300153,6,Morning,71146,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stops,300153,3,one,250863,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrival_time,300153,6,Night,91538,NaN,NaN,NaN,NaN,NaN,NaN,NaN
destination_city,300153,6,Mumbai,59097,NaN,NaN,NaN,NaN,NaN,NaN,NaN
class,300153,2,Economy,206666,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,"300,153.00",NaN,NaN,NaN,12.22,7.19,0.83,6.83,11.25,16.17,49.83


In [ ]:
# Cleaning
df = df.drop(columns=[c for c in ['index', 'Unnamed: 0'] if c in df.columns])
df.columns = df.columns.str.strip().str.lower()
for c in df.select_dtypes('object').columns:
    df[c] = df[c].str.strip()
df = df.dropna()
df = df[(df.price > 0) & (df.duration > 0)]
print('Clean shape:', df.shape)

Clean shape: (300153, 11)


## STEP 3 – Feature engineering (the "business" columns)
Each row = one ticket listing, so **Revenue = sum of price** (a standard proxy for this dataset).

In [ ]:
df['route'] = df.source_city + ' -> ' + df.destination_city
df['stops_label'] = df.stops.map({'zero': 0, 'one': 1, 'two_or_more': 2}).fillna(df.stops)
df['booking_window'] = pd.cut(df.days_left, [0, 3, 7, 14, 21, 30, 50],
                              labels=['0-3d (last minute)', '4-7d', '8-14d', '15-21d', '22-30d', '31-49d (early)'])
df['revenue'] = df.price
df['price_per_hour'] = df.price / df.duration
print(df[['route', 'stops_label', 'booking_window', 'revenue']].head())

             route  stops_label      booking_window  revenue
0  Delhi -> Mumbai            0  0-3d (last minute)     5953
1  Delhi -> Mumbai            0  0-3d (last minute)     5953
2  Delhi -> Mumbai            0  0-3d (last minute)     5956
3  Delhi -> Mumbai            0  0-3d (last minute)     5955
4  Delhi -> Mumbai            0  0-3d (last minute)     5955


## STEP 4 – Executive KPIs (same as Power BI Page 1)

In [ ]:
kpi = {
    'Total Tickets (Count of Flight)': len(df),
    'Total Revenue (₹)': df.revenue.sum(),
    'Average Price (₹)': df.price.mean(),
    'Average Duration (Hrs)': df.duration.mean(),
    'Number of Routes': df.route.nunique(),
    'Number of Airlines': df.airline.nunique(),
}
for k, v in kpi.items():
    print(f'{k:35s}: {v:,.2f}')

Total Tickets (Count of Flight)    : 300,153.00
Total Revenue (₹)                  : 6,270,094,275.00
Average Price (₹)                  : 20,889.66
Average Duration (Hrs)             : 12.22
Number of Routes                   : 30.00
Number of Airlines                 : 6.00


## STEP 5 – BI Dashboard 1: Executive Overview

In [ ]:
# Avg price vs days left
d = df.groupby('days_left', as_index=False).price.mean()
fig = px.area(d, x='days_left', y='price', title='Average Price vs Days Left',
              color_discrete_sequence=['#6f3fc4'])
fig.show()

# Top 5 routes by revenue
r = df.groupby('route', as_index=False).revenue.sum().nlargest(5, 'revenue')
px.bar(r.sort_values('revenue'), x='revenue', y='route', orientation='h',
       title='Top 5 Routes by Total Revenue', color_discrete_sequence=['#6f3fc4']).show()

# Avg price & count by airline
a = df.groupby('airline').agg(avg_price=('price', 'mean'), flights=('price', 'size')).reset_index()
fig = go.Figure()
fig.add_bar(x=a.airline, y=a.avg_price, name='Avg Price', marker_color='#6f3fc4')
fig.add_bar(x=a.airline, y=a.flights, name='Count of Flights', marker_color='#1e88ff', yaxis='y2')
fig.update_layout(title='Avg Price and Flight Count by Airline',
                  yaxis2=dict(overlaying='y', side='right'), barmode='group')
fig.show()

## STEP 6 – BI Dashboard 2: Route & Pricing Deep Dive

In [ ]:
# Price vs days left by stops
d = df.groupby(['days_left', 'stops_label'], as_index=False).price.mean()
px.line(d, x='days_left', y='price', color=d.stops_label.astype(str),
        title='Avg Price vs Days Left by Stops').show()

# Price vs duration
px.scatter(df.sample(min(20000, len(df)), random_state=1), x='duration', y='price',
           opacity=.4, title='Price vs Flight Duration').show()

# Price by departure time
order = ['Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late_Night']
d = df.groupby('departure_time', as_index=False).price.mean()
px.bar(d, x='departure_time', y='price', category_orders={'departure_time': order},
       title='Avg Price by Departure Time').show()

# Source x Destination heatmap
pv = df.pivot_table(index='source_city', columns='destination_city', values='price', aggfunc='mean')
px.imshow(pv, text_auto='.0f', color_continuous_scale='Purples',
          title='Avg Price: Source vs Destination').show()

# Route details table
df.groupby(['source_city', 'destination_city']).agg(
    avg_price=('price', 'mean'), avg_duration=('duration', 'mean'), tickets=('price', 'size')
).reset_index().sort_values('avg_duration', ascending=False).head(10)

,source_city,destination_city,avg_price,avg_duration,tickets
21,Kolkata,Chennai,"23,660.36",14.77,6653
8,Chennai,Kolkata,"22,669.93",14.52,6983
0,Bangalore,Chennai,"23,321.85",14.48,6410
2,Bangalore,Hyderabad,"21,226.12",14.16,8928
5,Chennai,Bangalore,"25,081.85",13.95,6493
23,Kolkata,Hyderabad,"21,500.01",13.85,7897
20,Kolkata,Bangalore,"22,744.81",13.79,9824
18,Hyderabad,Kolkata,"20,823.89",13.54,7987
16,Hyderabad,Chennai,"21,848.07",13.29,6395
28,Mumbai,Hyderabad,"21,004.05",13.26,10470


## STEP 7 – BI Dashboard 3: Airline & Class Performance

In [ ]:
px.bar(df.groupby(['airline', 'class'], as_index=False).price.mean(), x='airline', y='price',
       color='class', barmode='group', title='Avg Price by Airline and Class').show()
px.bar(df.groupby(['stops_label', 'class'], as_index=False).price.mean(), x='stops_label', y='price',
       color='class', barmode='group', title='Avg Price by Stops and Class').show()
px.bar(df.groupby(['airline', 'class'], as_index=False).size(), x='airline', y='size',
       color='class', title='Count of Flights by Airline and Class').show()

## STEP 8 – Revenue Intelligence (the "airport revenue" part)
These answer real business questions a revenue manager asks.

In [ ]:
# Q1: Which airline & class generate the most revenue?
rev = df.groupby(['airline', 'class'], as_index=False).revenue.sum()
rev['share_%'] = rev.revenue / rev.revenue.sum() * 100
display(rev.sort_values('revenue', ascending=False))
px.sunburst(rev, path=['class', 'airline'], values='revenue', title='Revenue Share: Class → Airline').show()

# Q2: Revenue by booking window (when do people pay the most?)
bw = df.groupby('booking_window', as_index=False).agg(avg_price=('price', 'mean'), revenue=('revenue', 'sum'))
display(bw)

# Q3: Last-minute premium: how much more do last-minute (<=3d) tickets cost vs early (>=31d)?
eco = df[df['class'] == 'Economy']
late = eco[eco.days_left <= 3].price.mean(); early = eco[eco.days_left >= 31].price.mean()
print(f'Economy last-minute premium: {(late/early-1)*100:.1f}%  (₹{late:,.0f} vs ₹{early:,.0f})')

# Q4: Most profitable routes per hour of flying
rt = df.groupby('route').agg(avg_price=('price', 'mean'), dur=('duration', 'mean'), revenue=('revenue', 'sum')).reset_index()
rt['revenue_per_hour'] = rt.avg_price / rt.dur
rt.sort_values('revenue_per_hour', ascending=False).head(10)

,airline,class,revenue,share_%
6,Vistara,Business,3361297636,53.61
1,Air_India,Business,1550516928,24.73
7,Vistara,Economy,525173099,8.38
2,Air_India,Economy,351012862,5.60
4,Indigo,Economy,229580207,3.66
3,GO_FIRST,Economy,130973972,2.09
0,AirAsia,Economy,65858089,1.05
5,SpiceJet,Economy,55681482,0.89


,booking_window,avg_price,revenue
0,0-3d (last minute),"28,068.68",286328624
1,4-7d,"25,698.24",563099880
2,8-14d,"23,768.36",1017404770
3,15-21d,"20,201.01",912095620
4,22-30d,"19,647.45",1147862712
5,31-49d (early),"19,260.92",2343302669


Economy last-minute premium: 180.7%  (₹13,808 vs ₹4,919)


,route,avg_price,dur,revenue,revenue_per_hour
4,Bangalore -> Mumbai,"23,128.62",10.91,299261197,"2,120.90"
25,Mumbai -> Bangalore,"23,147.87",11.61,298260354,"1,993.44"
27,Mumbai -> Delhi,"18,725.32",9.82,277303264,"1,907.23"
14,Delhi -> Mumbai,"19,355.83",10.37,295931282,"1,866.92"
9,Chennai -> Mumbai,"22,765.85",12.37,212587504,"1,839.72"
1,Bangalore -> Delhi,"17,723.31",9.78,243801907,"1,812.21"
26,Mumbai -> Chennai,"22,781.90",12.67,230780638,"1,798.68"
5,Chennai -> Bangalore,"25,081.85",13.95,162856455,"1,797.65"
3,Bangalore -> Kolkata,"23,500.06",13.10,235658614,"1,794.02"
15,Hyderabad -> Bangalore,"21,347.18",12.09,167660736,"1,765.20"


## STEP 9 – 🤖 ADVANCED: Ticket price prediction (Machine Learning)
Predicts price from airline, route, class, stops, timing and days left → powers **dynamic pricing**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

cat = ['airline', 'source_city', 'destination_city', 'departure_time', 'arrival_time', 'stops', 'class']
num = ['duration', 'days_left']
X, y = df[cat + num], df.price
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42)

pre = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)], remainder='passthrough')
model = Pipeline([('pre', pre), ('m', HistGradientBoostingRegressor(max_iter=300, learning_rate=.1, random_state=42))])
model.fit(Xtr, ytr)
pred = model.predict(Xte)
print(f'R²  : {r2_score(yte, pred):.3f}')
print(f'MAE : ₹{mean_absolute_error(yte, pred):,.0f}')

px.scatter(x=yte[:3000], y=pred[:3000], labels={'x': 'Actual', 'y': 'Predicted'}, title='Actual vs Predicted Price', opacity=.4).show()

R²  : 0.977
MAE : ₹2,002


In [ ]:
# Which features drive price? (permutation importance)
from sklearn.inspection import permutation_importance
idx = np.random.RandomState(0).choice(len(Xte), min(5000, len(Xte)), replace=False)
imp = permutation_importance(model, Xte.iloc[idx], yte.iloc[idx], n_repeats=3, random_state=0)
imp = pd.Series(imp.importances_mean, index=Xte.columns).sort_values()
px.bar(imp, orientation='h', title='Feature Importance (what drives ticket price)').show()

In [ ]:
# What-if tool for dynamic pricing
def predict_price(airline, source, dest, dep, arr, stops, cls, duration, days_left):
    row = pd.DataFrame([dict(airline=airline, source_city=source, destination_city=dest, departure_time=dep,
                             arrival_time=arr, stops=stops, **{'class': cls}, duration=duration, days_left=days_left)])
    return float(model.predict(row[cat + num])[0])

# Same flight, different booking time
for d in [1, 7, 15, 30, 45]:
    print(f'Delhi→Mumbai Vistara Economy, {d:>2} days left: ₹{predict_price("Vistara","Delhi","Mumbai","Morning","Evening","zero","Economy",2.2,d):,.0f}')

Delhi→Mumbai Vistara Economy,  1 days left: ₹8,987
Delhi→Mumbai Vistara Economy,  7 days left: ₹7,306
Delhi→Mumbai Vistara Economy, 15 days left: ₹5,527
Delhi→Mumbai Vistara Economy, 30 days left: ₹3,138
Delhi→Mumbai Vistara Economy, 45 days left: ₹3,010


## STEP 10 – 🤖 ADVANCED: Route segmentation (KMeans clustering)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
rt = df.groupby('route').agg(avg_price=('price', 'mean'), avg_duration=('duration', 'mean'),
                             tickets=('price', 'size'), revenue=('revenue', 'sum')).reset_index()
Z = StandardScaler().fit_transform(rt[['avg_price', 'avg_duration', 'tickets', 'revenue']])
rt['segment'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(Z).astype(str)
px.scatter(rt, x='avg_price', y='revenue', size='tickets', color='segment', hover_name='route',
           title='Route Segments (high / mid / low value)').show()
rt.groupby('segment')[['avg_price', 'avg_duration', 'tickets', 'revenue']].mean()

,avg_price,avg_duration,tickets,revenue
segment,,,,
0,"18,613.09",11.08,"11,683.30","217,671,853.80"
1,"22,307.06",13.78,"7,170.30","159,282,996.80"
2,"22,409.76",12.63,"11,161.70","250,054,576.90"


## STEP 11 – 🤖 ADVANCED: Pricing anomaly detection

In [ ]:
g = df.groupby(['route', 'airline', 'class']).price
df['z'] = (df.price - g.transform('mean')) / g.transform('std')
anom = df[df.z.abs() > 3]
print(f'{len(anom):,} unusually priced tickets ({len(anom)/len(df)*100:.2f}%)')
anom[['airline', 'route', 'class', 'days_left', 'price', 'z']].sort_values('z', ascending=False).head(10)

4,901 unusually priced tickets (1.63%)


,airline,route,class,days_left,price,z
158666,GO_FIRST,Hyderabad -> Mumbai,Economy,2,32803,15.15
158735,GO_FIRST,Hyderabad -> Mumbai,Economy,3,29129,13.18
94386,Air_India,Bangalore -> Mumbai,Economy,3,40354,11.30
127662,GO_FIRST,Kolkata -> Mumbai,Economy,2,25740,9.97
128729,AirAsia,Kolkata -> Mumbai,Economy,10,21087,9.85
102825,GO_FIRST,Bangalore -> Kolkata,Economy,1,27620,9.69
1461,AirAsia,Delhi -> Mumbai,Economy,9,29501,9.25
120035,Indigo,Kolkata -> Delhi,Economy,4,31952,9.07
53031,GO_FIRST,Mumbai -> Bangalore,Economy,2,26891,9.01
61592,GO_FIRST,Mumbai -> Kolkata,Economy,1,16917,8.97


## STEP 12 – Export for Power BI / Tableau
Creates a **star schema** (1 fact table + dimension tables) – the professional BI way.

In [ ]:
os.makedirs('bi_export', exist_ok=True)
df.to_csv('bi_export/flights_clean.csv', index=False)

fact = df.reset_index(drop=True).copy()
fact['ticket_id'] = fact.index + 1
dims = {
    'dim_airline': df[['airline']].drop_duplicates(),
    'dim_route': df[['route', 'source_city', 'destination_city']].drop_duplicates(),
    'dim_class': df[['class']].drop_duplicates(),
}
for n, t in dims.items():
    t.to_csv(f'bi_export/{n}.csv', index=False)
fact[['ticket_id', 'airline', 'route', 'class', 'departure_time', 'arrival_time', 'stops_label',
      'days_left', 'duration', 'price']].to_csv('bi_export/fact_tickets.csv', index=False)

import shutil; shutil.make_archive('bi_export', 'zip', 'bi_export')
from google.colab import files
files.download('bi_export.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## STEP 13 – Live web dashboard with Streamlit (optional, looks great in viva)

In [ ]:
%%writefile app.py
import streamlit as st, pandas as pd, plotly.express as px
st.set_page_config(page_title='Airline Revenue Dashboard', layout='wide')
df = pd.read_csv('bi_export/flights_clean.csv')
st.title('✈️ Airline Revenue Intelligence Dashboard')

st.sidebar.header('Filters')
air = st.sidebar.multiselect('Airline', sorted(df.airline.unique()), sorted(df.airline.unique()))
cls = st.sidebar.multiselect('Class', sorted(df['class'].unique()), sorted(df['class'].unique()))
src = st.sidebar.multiselect('Source', sorted(df.source_city.unique()), sorted(df.source_city.unique()))
dl = st.sidebar.slider('Days left', 1, 49, (1, 49))
f = df[df.airline.isin(air) & df['class'].isin(cls) & df.source_city.isin(src) & df.days_left.between(*dl)]

a, b, c, d = st.columns(4)
a.metric('Tickets', f'{len(f):,}'); b.metric('Revenue (₹)', f'{f.price.sum()/1e6:,.1f}M')
c.metric('Avg Price (₹)', f'{f.price.mean():,.0f}'); d.metric('Avg Duration (h)', f'{f.duration.mean():.2f}')

l, r = st.columns(2)
l.plotly_chart(px.area(f.groupby('days_left', as_index=False).price.mean(), x='days_left', y='price', title='Price vs Days Left'), use_container_width=True)
r.plotly_chart(px.bar(f.groupby('route', as_index=False).price.sum().nlargest(5, 'price'), x='price', y='route', orientation='h', title='Top 5 Routes by Revenue'), use_container_width=True)
l.plotly_chart(px.bar(f.groupby(['airline', 'class'], as_index=False).price.mean(), x='airline', y='price', color='class', barmode='group', title='Avg Price by Airline & Class'), use_container_width=True)
r.plotly_chart(px.imshow(f.pivot_table(index='source_city', columns='destination_city', values='price'), text_auto='.0f', title='Source vs Destination'), use_container_width=True)

Writing app.py


In [ ]:
# Run the dashboard: click the URL printed below; password = the IP address shown first
!npm install -q localtunnel
!curl -s ipv4.icanhazip.com
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501

⠙⠹⠸⠼⠴
up to date, audited 23 packages in 927ms
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠦136.87.118.110
⠙your url is: https://big-tools-lose.loca.lt
^C


In [ ]:

# Install LocalTunnel
!npm install -g localtunnel

# Start Streamlit in the background
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/logs.txt 2>&1 &

# Wait for Streamlit to start
import time
time.sleep(5)

# Create a new public URL
!lt --port 8501


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 3s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙your url is: https://thin-jobs-worry.loca.lt


## ✅ Conclusion & Business Recommendations
1. **Business class** earns far more per ticket than Economy – protect premium inventory on high-demand routes.
2. **Last-minute bookings** carry a large price premium → raise fares progressively as departure nears.
3. **Long-duration and multi-stop** itineraries are priced higher → target them with promotions only where seats stay empty.
4. Use the **ML model** to suggest a price per route/day, and the **anomaly list** to catch mis-priced fares.
5. **Route segments** show where to add capacity (high-value) and where to run discounts (low-value).

**Future scope:** live airline APIs, passenger-demand forecasting (time-series), reinforcement-learning pricing, airport ancillary revenue (duty-free, parking, cargo).